# Feature Generator Demo

Quick demo of `generate_features_by_window` to generate aggregated features from transaction data.

In [ ]:
import pandas as pd
from caketool.feature import generate_features_by_window

## 1. Create sample data

In [ ]:
# Sample transaction data
df = pd.DataFrame({
    "user_id": ["u1", "u1", "u1", "u1", "u2", "u2", "u2"],
    "report_date": pd.to_datetime([
        "2024-01-01", "2024-01-05", "2024-01-10", "2024-01-12",
        "2024-01-02", "2024-01-08", "2024-01-11"
    ]),
    "fs_event_timestamp": pd.to_datetime(["2024-01-15"] * 7),
    "amount": [100.0, 200.0, 150.0, 300.0, 50.0, 75.0, 120.0],
    "category": ["food", "transport", "food", "shopping", "food", "transport", "food"],
    "is_weekend": [False, False, False, False, False, False, False],
})
df

## 2. Generate basic features (lifetime)

In [ ]:
# Generate lifetime features for numeric columns
features = generate_features_by_window(
    df,
    client_id_col="user_id",
    report_date_col="report_date",
    fs_event_timestamp="fs_event_timestamp",
    numeric_cols=("amount",),
    backend="pandas",
)
features

## 3. Generate features with multiple lookback windows

In [ ]:
# Generate features with lookback 0 (lifetime), 7 days, 14 days
features = generate_features_by_window(
    df,
    client_id_col="user_id",
    report_date_col="report_date",
    fs_event_timestamp="fs_event_timestamp",
    lookback_days=(0, 7, 14),
    numeric_cols=("amount",),
    string_cols=("category",),
    boolean_cols=("is_weekend",),
    backend="pandas",
)
features.T  # Transpose for better readability

## 4. Generate features grouped by category

In [ ]:
# Pivot features by category (food, transport, shopping)
features = generate_features_by_window(
    df,
    client_id_col="user_id",
    report_date_col="report_date",
    fs_event_timestamp="fs_event_timestamp",
    key_cols=("category",),
    numeric_cols=("amount",),
    feature_prefix="txn",
    backend="pandas",
)
features.T

## 5. Generated Features Reference

| Column Type | Features Generated |
|-------------|-------------------|
| `numeric_cols` | min, max, avg, sum, std, cnt, diff, p25, p50, p75 |
| `string_cols` / `categorical_cols` | cnt, nunique, entropy |
| `boolean_cols` | poscnt, posratio |
| `date_cols` | firstdatediff, lastdatediff, daysbetween |